In [10]:
# Cell 1: Read the bronze_sles Delta table
df = spark.read.format('delta').load('Tables/dbo/bronze_sales')

# Show the first 10 rows
df.show()

StatementMeta(, dc34d0e0-f480-4ea7-be5d-45d2d167de09, 25, Finished, Available, Finished, False)

+---------+----------+----------------+------+---------------+--------+--------+---------+
|  OrderID| OrderDate|    CustomerName|Region|ProductCategory| Revenue|Quantity|   Status|
+---------+----------+----------------+------+---------------+--------+--------+---------+
|ORD-01255|18-08-2024|   Gaurav Mishra| South|      Furniture|     0.0|       3| Returned|
|ORD-01314|23-10-2023|     Aakash Bhat| South|      Furniture| 5911.62|       6|   Active|
|ORD-01042|29-08-2023|Rajesh Choudhury| South|Office Supplies|  899.44|      20|   Active|
|ORD-01709|08-01-2023|    Ajay Rastogi| North|Office Supplies|   74.82|      21|   Active|
|ORD-01484|04-11-2024|      Divya Kaur| North|Office Supplies| 1110.91|       7|   Active|
|ORD-01443|10-04-2024|    Geeta Pandey|  West|    Electronics|     0.0|       1|Cancelled|
|ORD-02004|22-08-2023|     Sneha Gupta|  East|      Furniture|15696.99|       7|   Active|
|ORD-01927|19-12-2023|     Arjun Reddy| South|Office Supplies| 3453.15|      13|   Active|

In [11]:
# Inspecting schema
df.printSchema()

StatementMeta(, dc34d0e0-f480-4ea7-be5d-45d2d167de09, 26, Finished, Available, Finished, False)

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Status: string (nullable = true)



In [12]:
# Cell 3: DataType of OrderDate from string to date
from pyspark.sql.functions import to_date

df = df.withColumn(
    'OrderDate', 
    to_date('OrderDate', 'dd-MM-yyyy')
)

df.printSchema()
df.show()

StatementMeta(, dc34d0e0-f480-4ea7-be5d-45d2d167de09, 27, Finished, Available, Finished, False)

root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Status: string (nullable = true)

+---------+----------+----------------+------+---------------+--------+--------+---------+
|  OrderID| OrderDate|    CustomerName|Region|ProductCategory| Revenue|Quantity|   Status|
+---------+----------+----------------+------+---------------+--------+--------+---------+
|ORD-01255|2024-08-18|   Gaurav Mishra| South|      Furniture|     0.0|       3| Returned|
|ORD-01314|2023-10-23|     Aakash Bhat| South|      Furniture| 5911.62|       6|   Active|
|ORD-01042|2023-08-29|Rajesh Choudhury| South|Office Supplies|  899.44|      20|   Active|
|ORD-01709|2023-01-08|    Ajay Rastogi| North|Office Supplies|   74.82|      21|   Active|
|ORD-01484|2024-11-04|      Divy

In [13]:
# Cell 4: Transformation 1 - Filter rows
# Keep only active orders with positive revenue
df_filtered = df.filter(
    (df['Status'] == 'Active') & (df['Revenue'] > 0)
)

# Check how many rows remain
print(f'Original row count: {df.count()}')
print(f'Filtered row count: {df_filtered.count()}')

StatementMeta(, dc34d0e0-f480-4ea7-be5d-45d2d167de09, 28, Finished, Available, Finished, False)

Original row count: 1525
Filtered row count: 1268


In [14]:
# Cell 5: Transformation 2 - Rename columns to snake case
df_renamed = df_filtered \
    .withColumnRenamed('OrderId', 'order_id') \
    .withColumnRenamed('OrderDate', 'order_date') \
    .withColumnRenamed('CustomerName', 'customer_name') \
    .withColumnRenamed('Region', 'region') \
    .withColumnRenamed('ProductCategory', 'product_category') \
    .withColumnRenamed('Revenue', 'revenue') \
    .withColumnRenamed('Quality', 'quality') \
    .withColumnRenamed('Status', 'status')

display(df_renamed.limit(5))
    

StatementMeta(, dc34d0e0-f480-4ea7-be5d-45d2d167de09, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe38bdef-9986-4de5-8e04-e0a0e38cd9f1)

In [15]:
# Cell 6: Transformation 3 - Add calculated column
from pyspark.sql.functions import col, round

# Apply a fixed INR to USD conversion rate (example: 1 USD = 90 INR)
EXCHANGE_RATE = 90.0

df_transformed = df_renamed.withColumn(
    'revenue_usd',
    round(col('revenue') / EXCHANGE_RATE, 2)
)

display(df_transformed.select('order_id', 'revenue', 'revenue_usd').limit(10))

StatementMeta(, dc34d0e0-f480-4ea7-be5d-45d2d167de09, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bc08519d-8a10-400b-8383-4a2c9b74ff52)

In [16]:
# Cell 7: Write the transformed DataFrame as silver_sales Delta table
df_transformed.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('silver_sales')

StatementMeta(, dc34d0e0-f480-4ea7-be5d-45d2d167de09, 31, Finished, Available, Finished, False)